In [1]:
# CO2 LSTM FORECASTER -PER COUNTRY(TRAIN–TEST)

import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input

#SEED
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# 1. LOAD DATA
file_path = "/kaggle/input/climate-and-energy-consumption-dataset-20202024/global_climate_energy_2020_2024.csv"
df = pd.read_csv(file_path)

df['date'] = pd.to_datetime(df['date'], dayfirst=True, errors='coerce')
df = df.dropna(subset=['date'])

df['country'] = df['country'].astype(str).str.strip().str.title()
df = df.sort_values(['country', 'date'])

# 2. FEATURES
FEATURES = [
    'co2_emission',
    'industrial_activity_index',
    'energy_consumption',
    'avg_temperature',
    'energy_price'
]

df['co2_next_day'] = df.groupby('country')['co2_emission'].shift(-1)

df = df[df.groupby('country')['co2_emission']
          .transform('count') - df.groupby('country').cumcount() > 1]

for col in FEATURES + ['co2_next_day']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df = df.dropna(subset=FEATURES + ['co2_next_day'])

# 3. CREATE SEQUENCES
WINDOW = 9

def create_sequences(df_country):
    if len(df_country) < WINDOW + 10:
        return None, None, None, None

    scaler_x = MinMaxScaler()
    scaler_y = MinMaxScaler()

    X_data = scaler_x.fit_transform(df_country[FEATURES])
    y_data = scaler_y.fit_transform(df_country[['co2_next_day']])

    X, y = [], []
    for i in range(len(X_data) - WINDOW):
        X.append(X_data[i:i + WINDOW])
        y.append(y_data[i + WINDOW])

    return np.array(X), np.array(y), scaler_x, scaler_y

# 4. PREPARE TRAIN–TEST DATA
country_models = {}
country_histories = {}
country_names = []

print("Preparing countries...")

for country in sorted(df['country'].unique()):
    df_c = df[df['country'] == country].copy()

    X, y, scaler_x, scaler_y = create_sequences(df_c)

    if X is None or len(X) < 50:
        print(f"{country}: skipped (insufficient data)")
        continue

    train_size = int(len(X) * 0.8)

    country_models[country] = {
        'X_train': X[:train_size],
        'y_train': y[:train_size],
        'X_test': X[train_size:],
        'y_test': y[train_size:],
        'scaler_x': scaler_x,
        'scaler_y': scaler_y,
        'df': df_c
    }

    country_names.append(country)
    print(f"{country}: Train={train_size}, Test={len(X)-train_size}")

# 5. MODEL
def build_lstm():
    model = Sequential([
        Input(shape=(WINDOW, len(FEATURES))),
        LSTM(50),
        Dropout(0.2),
        Dense(25, activation='relu'),
        Dropout(0.2),
        Dense(1)
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(0.0005),
        loss='mse',
        metrics=['mae']
    )
    return model

# 6. TRAIN
print("\nTraining models...\n")

train_accuracies = [] 
for country in country_names:
    print(f"Training {country}")

    model = build_lstm()
    data = country_models[country]

    history = model.fit(
        data['X_train'],
        data['y_train'],
        epochs=40,
        batch_size=32,
        verbose=1
    )

    country_models[country]['model'] = model
    country_histories[country] = history
    
    y_train_pred = model.predict(data['X_train'], verbose=0)
    y_train_pred_real = data['scaler_y'].inverse_transform(y_train_pred)
    y_train_real = data['scaler_y'].inverse_transform(data['y_train'])
    
    mae_train = mean_absolute_error(y_train_real, y_train_pred_real)
    mean_actual_train = np.mean(y_train_real)
    accuracy_train = max(0, 100 - (mae_train / mean_actual_train * 100))
    train_accuracies.append(accuracy_train)
    
    print(f"{country} Training Accuracy: {accuracy_train:.1f}%")

print(f"\nAverage Training Accuracy: {np.mean(train_accuracies):.1f}%") 

# 7. TESTING 
print("\n" + "="*60)
print("TEST RESULTS")
print("="*60)

accuracies = []

for country in country_names:
    data = country_models[country]
    model = data['model']

    y_pred = model.predict(data['X_test'], verbose=0)
    y_pred_real = data['scaler_y'].inverse_transform(y_pred)
    y_test_real = data['scaler_y'].inverse_transform(data['y_test'])

    mae = mean_absolute_error(y_test_real, y_pred_real)
    mean_actual = np.mean(y_test_real)

    accuracy = max(0, 100 - (mae / mean_actual * 100))
    accuracies.append(accuracy)

    print(f"{country:<20} MAE: {mae:>8.2f} | Accuracy: {accuracy:>6.1f}%")

print("\nAverage Test Accuracy:", np.mean(accuracies), "%")

# 8. DETAILED ONE-WINDOW TEST OUTPUT (PER COUNTRY)
for country in country_names:
    data = country_models[country]
    df_c = data['df']
    model = data['model']

    X_test = data['X_test']
    y_test = data['y_test']

    if len(X_test) == 0:
        continue

    i = -1  # last test window

    y_pred = model.predict(X_test, verbose=0)
    y_pred_real = data['scaler_y'].inverse_transform(y_pred).flatten()
    y_test_real = data['scaler_y'].inverse_transform(y_test).flatten()

    total_sequences = len(X_test) + len(data['X_train'])
    start_idx = len(df_c) - total_sequences - 1 + i
    window_df = df_c.iloc[start_idx:start_idx + WINDOW]
    target_row = df_c.iloc[start_idx + WINDOW]

    error = abs(y_pred_real[i] - y_test_real[i])
    error_pct = error / y_test_real[i] * 100
    accuracy = max(0, 100 - error_pct)


    print(f"\n--- {country} ---")
    print(f"Window dates: {window_df['date'].iloc[0].date()} to {window_df['date'].iloc[-1].date()}")
    print("Day | Date       | CO2    | Industrial | Energy | Temp  | Price")
    print("-"*75)

    for d, row in enumerate(window_df.itertuples(), start=1):
        print(f"{d:>3} | {row.date.date()} | {row.co2_emission:>6.1f} | "
              f"{row.industrial_activity_index:>10.2f} | {row.energy_consumption:>6.1f} | "
              f"{row.avg_temperature:>5.1f} | {row.energy_price:>6.1f}")

    print(f"Predicted day 10 CO2: {y_pred_real[i]:.2f}")
    print(f"Actual day 10 CO2:    {y_test_real[i]:.2f}")
    print(f"Error: {error:.2f} ({error_pct:.1f}%)")
    print(f"Accuracy: {accuracy:.1f}%")


2026-02-07 00:02:40.196105: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770422560.457443      17 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770422560.527032      17 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770422561.114602      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770422561.114668      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770422561.114671      17 computation_placer.cc:177] computation placer alr

Preparing countries...
Australia: Train=568, Test=142
Brazil: Train=568, Test=142
Canada: Train=568, Test=142
China: Train=568, Test=142
France: Train=568, Test=142
Germany: Train=568, Test=142
India: Train=568, Test=142
Indonesia: Train=568, Test=142
Italy: Train=568, Test=142
Japan: Train=568, Test=142
Mexico: Train=568, Test=142
Netherlands: Train=568, Test=142
Norway: Train=568, Test=142
Poland: Train=568, Test=142
South Africa: Train=568, Test=142
Spain: Train=568, Test=142
Sweden: Train=568, Test=142
Turkey: Train=568, Test=142
United Kingdom: Train=568, Test=142
United States: Train=568, Test=142

Training models...

Training Australia
Epoch 1/40


2026-02-07 00:02:56.590804: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - loss: 0.2124 - mae: 0.3878
Epoch 2/40
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0799 - mae: 0.2285
Epoch 3/40
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0779 - mae: 0.2311
Epoch 4/40
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0754 - mae: 0.2259
Epoch 5/40
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0789 - mae: 0.2294
Epoch 6/40
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0748 - mae: 0.2309
Epoch 7/40
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0681 - mae: 0.2132
Epoch 8/40
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0702 - mae: 0.2190
Epoch 9/40
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0696 - mae: 0.2230
Epoch 10/40
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0720 - mae: 0.2273
Epoch 11/40
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0660 - mae: 0.2131
Epoch 12/40
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0658 - mae: 0.2141
Epoch 13/40
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0